# Step 4: SHAP for every model

Explains the encoder of all 130 models with DeepExplainer and saves each one's importance: the mean |SHAP value| of
every gene for every node, over the explained samples (genes x nodes). Every model is explained on the same 100
background and 50 explained samples. Task N is model N for N < 125; tasks 125 to 129 are the eADAGEs of groups 0
to 4. Results that are already saved are skipped; `shap_all.slurm` runs one task per job.

Settings.

In [ ]:
import os, sys
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # hide TensorFlow's startup messages

PROJECT = os.path.expandvars("$SCRATCH/CSC-488-Code")    # project folder: data, lab code and results
DATA_PATH = os.path.join(PROJECT, "data", "Copy of ecoli_MG1655plus_genome_part2_log_counts_norm_01.csv")
SEQADAGE = os.path.join(PROJECT, "seqADAGE")              # the lab's code (my fork, Jordin branch)
RUN_DIR = os.path.join(PROJECT, "runs")                   # every notebook saves and reads its results here
os.chdir(PROJECT)

N_GROUPS, MODELS_PER_GROUP, BASE_SEED = 5, 25, 171       # 5 groups of 25 models, seeds 171 to 295

import json
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SHAP_SEED, N_BACKGROUND, N_EXPLAIN = 0, 100, 50
TASKS = "all"   # "all", or a list of task numbers
FORCE = False   # True = redo results that are already saved

# as a batch job, array task N explains only task N
IN_BATCH = "SLURM_ARRAY_TASK_ID" in os.environ
if IN_BATCH:
    TASKS = [int(os.environ["SLURM_ARRAY_TASK_ID"])]

The SHAP class: builds a saved model's encoder, h = tanh(xW + b), and explains it with DeepExplainer.

In [ ]:
class AdageShap:
    def __init__(self, path, background):
        """path: a saved single model or eADAGE (.npz); background: the background samples (samples x genes)."""
        import keras
        import shap
        import tensorflow as tf
        tf.get_logger().setLevel("ERROR")
        warnings.filterwarnings("ignore", message="The structure of `inputs` doesn't match")
        warnings.filterwarnings("ignore", message="Your TensorFlow version is newer than 2.4.0")
        with np.load(path) as z:
            W, b, act1 = z["W"].astype("float32"), z["eb"].astype("float32"), str(z["act1"])
        inp = keras.Input(shape=(W.shape[0],))
        dense = keras.layers.Dense(W.shape[1], activation=act1)
        self.encoder = keras.Model(inp, dense(inp))  # 6,390 gene values in, 450 node activations out
        dense.set_weights([W, b])
        self.explainer = shap.DeepExplainer(self.encoder, np.asarray(background, "float32"))

    def shap_values(self, samples):
        """Signed SHAP values, samples x genes x nodes."""
        sv = self.explainer.shap_values(np.asarray(samples, "float32"))
        return np.stack(sv, axis=-1) if isinstance(sv, list) else np.asarray(sv)

    @staticmethod
    def importance(values):
        """Mean absolute SHAP value over the explained samples, genes x nodes."""
        return np.abs(values).mean(axis=0).astype("float32")

Pick the samples, find the model for a task number, and explain one task.

In [ ]:
def pick_samples(n_samples):
    """Row numbers of the background and explained samples. They only depend on the seed, so every model is
    explained on the same samples."""
    perm = np.random.default_rng(SHAP_SEED).permutation(n_samples)
    return perm[:N_BACKGROUND], perm[N_BACKGROUND:N_BACKGROUND + N_EXPLAIN]


def target(task):
    """(name, model file, output file) for a task number."""
    n_models = N_GROUPS * MODELS_PER_GROUP
    if task < n_models:
        g, seed = task // MODELS_PER_GROUP, BASE_SEED + task
        return (f"model {task} (seed {seed})", os.path.join(RUN_DIR, "models", f"group{g}", f"model_seed{seed}.npz"),
                os.path.join(RUN_DIR, "shap", f"group{g}", f"shap_seed{seed}.npy"))
    g = task - n_models
    return (f"eADAGE of group {g}", os.path.join(RUN_DIR, "eadage", f"eadage_group{g}.npz"),
            os.path.join(RUN_DIR, "shap", f"group{g}", "shap_eadage.npy"))


def explain_one(X, task, force=False):
    import keras
    name, src, out = target(task)
    if os.path.exists(out) and not force:
        print(f"{name}: already saved, skipping")
        return
    bg, ex = pick_samples(len(X))
    start = time.time()
    s = AdageShap(src, X[bg])
    imp = s.importance(s.shap_values(X[ex]))
    secs = time.time() - start
    os.makedirs(os.path.dirname(out), exist_ok=True)
    tmp = out.replace(".npy", f".tmp{os.getpid()}.npy")  # write, then rename, so a job cut off leaves no half file
    np.save(tmp, imp)
    os.replace(tmp, out)
    with open(out.replace(".npy", ".json"), "w") as f:
        json.dump(dict(task=task, name=name, model=src, seconds=round(secs, 1)), f, indent=2)
    print(f"{name}: importance {imp.shape} in {secs:.0f}s")
    keras.backend.clear_session()

Explain every model.

In [ ]:
X = pd.read_csv(DATA_PATH, index_col=0).values.T.astype("float32")  # samples x genes
for task in (range(N_GROUPS * MODELS_PER_GROUP + N_GROUPS) if TASKS == "all" else TASKS):
    explain_one(X, task, FORCE)

How many of the background and explained samples were also training samples (Keras held out the last 10% of samples)
(thesis Section 5.6), and the time per model.

In [ ]:
if not IN_BATCH:
    bg, ex = pick_samples(len(X))
    first_held_out = int(np.floor(len(X) * 0.9))
    print(f"training samples: {int((bg < first_held_out).sum())} of {N_BACKGROUND} background, "
          f"{int((ex < first_held_out).sum())} of {N_EXPLAIN} explained")
    secs = [json.load(open(target(t)[2].replace(".npy", ".json")))["seconds"]
            for t in range(N_GROUPS * MODELS_PER_GROUP + N_GROUPS)]
    print(f"time per model: median {np.median(secs) / 60:.1f} min")

Total SHAP value of each node (summed over genes), in single models and eADAGEs (thesis Figure 8E).

In [ ]:
if not IN_BATCH:
    n_models = N_GROUPS * MODELS_PER_GROUP
    totals = {kind: np.concatenate([np.load(target(t)[2]).sum(axis=0) for t in tasks])
              for kind, tasks in [("single models", range(n_models)), ("eADAGEs", range(n_models, n_models + N_GROUPS))]}
    fig, ax = plt.subplots(figsize=(5.5, 3.6))
    for (kind, t), color in zip(totals.items(), ["#2a78d6", "#eb6834"]):
        ax.hist(t, bins=60, histtype="step", lw=2, color=color, density=True, label=kind)
    ax.set(title="per node: total SHAP (summed over genes)", xlabel="sum of mean |SHAP value|", ylabel="density")
    ax.legend(frameon=False)
    plt.tight_layout()
    fig.savefig(os.path.join(RUN_DIR, "shap", "shap_totals.png"), dpi=120)
    plt.show()